****Petran van Rens****

Exercise Hierarchical models and testing

****Choice of model****

The model chosen was a binomial model, given that people either reuse their towel or they do not. This looks as follows:

Y_ij ~ Bin(n_ij,p_ij)

Response variable Y_ij is the amount of customers who reuse their towels, out of the total amount of customers n_ij.

    here i is the study (1-7) and j is the group (control / social intervention)


****Heterogeneity****

Since the studies all have different very different baselines, we use 

logit(*p_ij*)= *α_i* + β*X_ij

    where *α_i* ~ N(mean_α, sigma_study)
    sigma_study indicates the variance of the baseline values, with a high value meaning that the studies are very heterogeneous.

In [56]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import bambi as bmb
import arviz as az

In [57]:
# read in data
data_file = "towelData.csv"
data = pd.read_csv(data_file, sep=';', encoding='latin1')
count = data.iloc[:, -1] # get the last column with numbers or yes/no

# count has the number of yes and no for control and social norm groups
control_yes = count[::4].to_numpy() # every 4th starting from 0 - control group + yes
control_no = count[2::4].to_numpy() # every 4th starting from 2 - control group + no
control_total = np.array([y + n for y, n in zip(control_yes, control_no)])

social_yes = count[1::4].to_numpy() # every 4th starting from 1 - social norm group + yes
social_no = count[3::4].to_numpy() # every 4th starting from 3 - social norm group + no
social_total = np.array([y + n for y, n in zip(social_yes, social_no)])

study = np.arange(1,len(control_yes)+1) # 7 diferent studies

control_data = pd.DataFrame({"reuse": control_yes, "total": control_total, "group": "control", "study": study})
social_data = pd.DataFrame({ "reuse": social_yes, "total": social_total, "group": "social", "study": study})

combined_data = pd.concat([control_data, social_data], ignore_index=True)

# Convert data types - important for bambi that they are correctly set 
# can give errors otherwise
combined_data['reuse'] = combined_data['reuse'].astype(int)
combined_data['total'] = combined_data['total'].astype(int)
combined_data['group'] = combined_data['group'].astype('category')
combined_data['study'] = combined_data['study'].astype('category')
combined_data["reuse_rate"] = (    combined_data["reuse"] / combined_data["total"])



In [58]:
model = bmb.Model(
    "p(reuse, total) ~ group + (1 | study)",
    combined_data,
    family="binomial"
)

In [59]:
results = model.fit(target_accept=0.95)
summary = az.summary(
    results,
    ci_prob=0.90
)
summary

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, group, 1|study_sigma, 1|study_offset]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 12 seconds.


,mean,sd,eti90_lb,eti90_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
Intercept,0.4,0.41,-0.25,1.1,957,1242,1.00,0.013,0.011
group[social],0.209,0.078,0.079,0.34,1984,2016,1.00,0.0017,0.0014
1|study_sigma,1.13,0.42,0.64,1.9,723,1132,1.00,0.015,0.014
1|study[1],-0.92,0.42,-1.6,-0.26,980,1272,1.00,0.013,0.011
1|study[2],-0.85,0.41,-1.5,-0.19,941,1170,1.00,0.014,0.011
1|study[3],-0.12,0.41,-0.78,0.53,947,1214,1.00,0.013,0.011
1|study[4],-0.62,0.41,-1.3,0.035,946,1228,1.00,0.014,0.011
1|study[5],1.15,0.53,0.31,2,1248,1545,1.00,0.015,0.011
1|study[6],0.96,0.42,0.29,1.6,966,1274,1.00,0.013,0.011
1|study[7],0.78,0.44,0.083,1.5,1024,1404,1.00,0.014,0.01


**Parameter values**

According to the results, the posterior mean (β) is 0.209, with confidence intervals of 90% [0.078,0.34].

Sigma_study is 1.1, with confidence intervals [0.63,1.8]
This indicates that there is a large heterogeneity in baseline log-odds between the studies.
This was expected, since values for *p* range from 0.35 to 0.93.
be
The posterior intercept mean is 0.4, which relates to approximately *p ≈ 0.60*



**Hypotheses** 

H_0: β ≤ 0

H_1: β > 0


Since 0 ∉ [0.078,0.34], the null hypothesis is rejected. We can say that people are more likely to reuse their towel, if given a social reminder.

 
